<a id="lambert"></a>

$$
\text{朗伯余弦定律 (Lambert's Cosine Law): 漫反射的亮度正比于 } \cos\theta = n \cdot l \\ \quad \\
\text{推导: 一束平行光, 横截面积为 } A_{beam} \text{, 携带的光通量 (功率) 为 } \Phi \\
\text{照到一个法向量为 n 的表面上, l 是指向光源的单位向量, } \theta \text{ 是 n 和 l 的夹角} \\
\text{表面上被照亮的面积比横截面大: 横截面是表面在垂直于光线方向上的投影, } A_{beam} = A_{surf} \cos\theta \\
\text{单位面积接收的功率 (辐照度 Irradiance): } E = \frac{\Phi}{A_{surf}} = \frac{\Phi}{A_{beam}} \cos\theta = E_0 \, (n \cdot l) \\
\text{(冬天太阳斜射, 同样的阳光铺在更大的地面上, 所以更冷)} \\ \quad \\
\text{理想漫反射表面 (Lambertian Surface) 把接收到的光均匀地向各个方向散射, 所以从哪个方向看亮度都一样:} \\
I_{diffuse} = k_d \, I_{light} \max(0, \ n \cdot l) \quad (\text{背光面 } n \cdot l < 0 \text{ 截断为 0})
$$

数学基础: [投影 (向量在某方向上的分量)](../math_basics/linear_algebra.ipynb#projection)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

np.set_printoptions(suppress=True, precision=4)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

def normalize(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

# 一个单位正方形 (面积 1) 绕 x 轴倾斜 theta, 计算它在垂直于光线方向上的投影面积
l = np.array([0.0, 1.0, 0.0])                          # 光从正上方照下来
for theta_deg in [0, 30, 60, 80]:
    th = np.deg2rad(theta_deg)
    e1 = np.array([1.0, 0.0, 0.0])
    e2 = np.array([0.0, np.sin(th), np.cos(th)])     # 正方形的两条边
    n = np.cross(e2, e1)                               # 法向量: 和光线的夹角就是 theta
    # 投影到垂直于 l 的平面: 去掉 l 方向的分量, 投影面积 = 两条投影边的叉积长度
    e1_p, e2_p = e1 - (e1 @ l) * l, e2 - (e2 @ l) * l
    proj_area = np.linalg.norm(np.cross(e1_p, e2_p))
    # 蒙特卡洛: 在光束横截面 [-1, 1]^2 上均匀撒光线, 统计打在正方形上的比例
    rng = np.random.default_rng(theta_deg)
    rays_xz = rng.uniform(-1, 1, size=(200000, 2))
    # 光线 (x, y, z) = (x0, s, z0) 沿 -y 方向, 和正方形所在平面 n . p = 0 求交, 再用边的坐标判断是否在正方形内
    if abs(n @ l) > 1e-12:
        s = -(n[0] * rays_xz[:, 0] + n[2] * rays_xz[:, 1]) / n[1]
        pts = np.column_stack([rays_xz[:, 0], s, rays_xz[:, 1]])
        a1, a2 = pts @ e1, pts @ e2
        hit_fraction = np.mean((a1 >= 0) & (a1 <= 1) & (a2 >= 0) & (a2 <= 1))
    else:
        hit_fraction = 0.0
    print(f'theta = {theta_deg:>2} deg: 投影面积 = {proj_area:.4f}, 蒙特卡洛横截面积 = {hit_fraction * 4:.4f}, cos(theta) = {np.cos(th):.4f}, n.l = {n @ l:.4f}')

<a id="reflection-vector"></a>

$$
\text{反射向量 (Reflection Vector): } r = 2 (n \cdot l) \, n - l \\ \quad \\
\text{推导: 把指向光源的 l 分解成平行于法向量和垂直于法向量的两部分 (投影)} \\
l_\parallel = (n \cdot l) \, n, \quad l_\perp = l - (n \cdot l) \, n \\
\text{镜面反射: 法向分量不变, 切向分量反向 (入射角 = 反射角)} \\
r = l_\parallel - l_\perp = (n \cdot l) n - l + (n \cdot l) n = 2 (n \cdot l) \, n - l \\ \quad \\
\text{检查: } n \cdot r = 2(n \cdot l) - n \cdot l = n \cdot l \text{ (和法向量夹角相同)}, \quad |r|^2 = 4 (n \cdot l)^2 - 4 (n \cdot l)^2 + 1 = 1 \\
\text{矩阵形式: } r = (2 n n^T - I) \, l \text{, 其中 } H = I - 2 n n^T \text{ 是 Householder 反射矩阵 (关于以 n 为法向量的平面做镜像)} \\
H^T H = I - 4 n n^T + 4 n (n^T n) n^T = I \text{ (正交矩阵), } \det H = -1 \text{ (反射)} \\
\text{光线追踪里用的是入射方向 d (指向表面, } d = -l \text{), 代入得 } r = d - 2 (d \cdot n) \, n
$$

数学基础: [投影矩阵](../math_basics/linear_algebra.ipynb#projection) · [正交矩阵](../math_basics/linear_algebra.ipynb#orthogonal) · [行列式](../math_basics/linear_algebra.ipynb#determinant)

In [ ]:
def reflect(l, n):
    return 2 * np.sum(n * l, axis=-1, keepdims=True) * n - l

rng = np.random.default_rng(1)
n_r = normalize(rng.normal(size=3))
l_r = normalize(rng.normal(size=3))
r_r = reflect(l_r, n_r)
print(f'|r| = {np.linalg.norm(r_r):.6f}, n.l = {n_r @ l_r:.6f}, n.r = {n_r @ r_r:.6f}')
print(f'n, l, r 共面 (三者组成的行列式) = {np.linalg.det(np.column_stack([n_r, l_r, r_r])):.2e}')
print(f'(r + l) 平行于 n ? {np.allclose(np.cross(r_r + l_r, n_r), 0)}')

H = np.eye(3) - 2 * np.outer(n_r, n_r)
print(f'r == -H l ? {np.allclose(r_r, -H @ l_r)}, H^T H == I ? {np.allclose(H.T @ H, np.eye(3))}, det H = {np.linalg.det(H):.4f}')

d_in = -l_r
print(f'入射方向形式 d - 2 (d.n) n == r ? {np.allclose(d_in - 2 * (d_in @ n_r) * n_r, r_r)}')

<a id="phong"></a>

$$
\text{冯氏光照模型 (Phong Reflection Model): 环境光 + 漫反射 + 镜面高光} \\
I = k_a I_a + k_d I_l \max(0, n \cdot l) + k_s I_l \max(0, r \cdot v)^{\alpha} \\
v \text{: 指向观察者的单位向量, } \alpha \text{: 光泽度 (shininess), 越大高光越小越锐利} \\
\text{(镜面反射的光集中在 r 方向附近, 观察方向越接近 r 越亮, } \cos^\alpha \text{ 是一个经验的衰减函数)} \\ \quad \\
\text{Blinn-Phong: 用半程向量 (Half Vector) 代替反射向量, 计算更便宜} \\
h = \frac{l + v}{|l + v|}, \quad I_{spec} = k_s I_l \max(0, n \cdot h)^{\alpha'} \\ \quad \\
\text{为什么等价: 当 } v = r \text{ (正好在镜面方向) 时, } l + v = l + r = 2 (n \cdot l) n \text{ 平行于 n, 所以 } h = n, \ n \cdot h = 1 \\
\text{当 n, l, v 共面时, 设 l 和 v 相对于 n 的有向角为 } \theta_l, \theta_v \text{, 则 r 的角度为 } -\theta_l \\
\angle(r, v) = |\theta_v + \theta_l|, \quad h \text{ 的角度 = } \frac{\theta_l + \theta_v}{2} \implies \angle(n, h) = \frac{|\theta_l + \theta_v|}{2} = \frac{1}{2} \angle(r, v) \\
\text{角度只有一半, 所以 } \alpha' \approx 4 \alpha \text{ 时高光大小相近 (} \cos^{\alpha}\phi \approx e^{-\alpha \phi^2 / 2} \text{, 把 } \phi \text{ 换成 } \phi/2 \text{ 需要 4 倍指数)}
$$

数学基础: [泰勒展开 (cos 的二阶近似)](../math_basics/calculus.ipynb#taylor)

In [ ]:
def unit2(angle):
    # 在 (x, y) 平面内, 相对于法向量 n = (0, 1, 0) 的有向角
    return np.array([np.sin(angle), np.cos(angle), 0.0])

n_up = np.array([0.0, 1.0, 0.0])
theta_l = np.deg2rad(35)
l_p = unit2(theta_l)
r_p = reflect(l_p, n_up)
for theta_v in np.deg2rad([-35, -20, 0, 15, 40]):
    v_p = unit2(theta_v)
    h_p = normalize(l_p + v_p)
    ang_rv = np.arccos(np.clip(r_p @ v_p, -1, 1))
    ang_nh = np.arccos(np.clip(n_up @ h_p, -1, 1))
    print(f'theta_v = {np.rad2deg(theta_v):6.1f} deg: angle(r, v) = {np.rad2deg(ang_rv):6.2f}, angle(n, h) = {np.rad2deg(ang_nh):6.2f} (一半)')

alpha = 20
phis = np.linspace(-np.pi / 2, np.pi / 2, 400)          # 观察方向偏离镜面方向的角度
phong = np.maximum(np.cos(phis), 0)**alpha
blinn_same = np.maximum(np.cos(phis / 2), 0)**alpha
blinn_4x = np.maximum(np.cos(phis / 2), 0)**(4 * alpha)
print(f'半高宽 (度): Phong = {np.rad2deg(np.ptp(phis[phong > 0.5])):.2f}, '
      f'Blinn (同指数) = {np.rad2deg(np.ptp(phis[blinn_same > 0.5])):.2f}, Blinn (4 倍指数) = {np.rad2deg(np.ptp(phis[blinn_4x > 0.5])):.2f}')

plt.figure(figsize=(4, 3))
plt.plot(np.rad2deg(phis), phong, label=f'Phong alpha={alpha}')
plt.plot(np.rad2deg(phis), blinn_same, '--', label=f'Blinn alpha={alpha}')
plt.plot(np.rad2deg(phis), blinn_4x, ':', label=f'Blinn alpha={4 * alpha}')
plt.xlabel('angle between v and r (deg)')
plt.ylabel('specular')
plot_nicely()
plt.show()

<a id="ray-intersection"></a>

$$
\text{光线求交 (Ray Intersection): 光线 } p(t) = o + t \, d, \ t > 0 \quad (o \text{ 起点, d 单位方向}) \\ \quad \\
\text{1. 光线与球 (Ray-Sphere): 球面 } |p - c|^2 = R^2 \text{, 代入光线, 令 } m = o - c \\
|m + t d|^2 = R^2 \implies (d \cdot d) \, t^2 + 2 (d \cdot m) \, t + (m \cdot m - R^2) = 0 \\
\text{一元二次方程 (d 是单位向量, a = 1): } b = d \cdot m, \ c' = m \cdot m - R^2, \ \Delta = b^2 - c' \\
t = -b \pm \sqrt{\Delta} \quad (\Delta < 0 \text{ 不相交}, \ \Delta = 0 \text{ 相切}, \ \Delta > 0 \text{ 两个交点}) \\
\text{取最小的正根 (起点在球内时较小的根为负, 取较大的那个)} \\
\text{交点处的法向量: } n = \frac{p - c}{R} \text{ (球心指向交点, 也就是 } |p - c|^2 \text{ 的梯度方向)} \\ \quad \\
\text{2. 光线与平面 (Ray-Plane): 平面 } n \cdot (p - p_0) = 0 \\
n \cdot (o + t d - p_0) = 0 \implies t = \frac{n \cdot (p_0 - o)}{n \cdot d} \quad (n \cdot d = 0 \text{ 时光线和平面平行})
$$

数学基础: [梯度垂直于等值面](../math_basics/calculus.ipynb#gradient) · [牛顿法 / 求根](../math_basics/calculus.ipynb#newton)

In [ ]:
def ray_sphere(O, D, center, radius, eps=1e-6):
    # O, D: (N, 3), 返回每条光线最近的正交点参数 t (不相交为 inf)
    m = O - center
    b = np.sum(D * m, axis=1)
    c = np.sum(m * m, axis=1) - radius**2
    disc = b**2 - c
    sq = np.sqrt(np.maximum(disc, 0))
    t1, t2 = -b - sq, -b + sq
    t = np.where(t1 > eps, t1, np.where(t2 > eps, t2, np.inf))
    return np.where(disc >= 0, t, np.inf)

def ray_plane(O, D, p0, normal, eps=1e-6):
    denom = D @ normal
    with np.errstate(divide='ignore', invalid='ignore'):
        t = ((p0 - O) @ normal) / denom
    return np.where((np.abs(denom) > 1e-12) & (t > eps), t, np.inf)

# 验证: 交点在球面上, 并且和 brentq 数值求根的结果一致
rng = np.random.default_rng(2)
center, radius = np.array([0.0, 1.0, -3.0]), 1.0
O = np.array([[0.0, 1.0, 2.0]] * 5)
D = normalize(np.array([0.0, 0.0, -1.0]) + 0.15 * rng.normal(size=(5, 3)))
t_hit = ray_sphere(O, D, center, radius)
for o_, d_, t_ in zip(O, D, t_hit):
    if np.isfinite(t_):
        f = lambda t: np.linalg.norm(o_ + t * d_ - center) - radius
        t_bq = brentq(f, 0, t_ + 1e-3) if f(0) > 0 else np.nan     # 从 0 到交点之间只有一个根 (第一个交点)
        print(f't = {t_:.8f}, brentq = {t_bq:.8f}, |p - c| = {np.linalg.norm(o_ + t_ * d_ - center):.8f}')
    else:
        print('没有交点')

# 起点在球内: 应该取较大的那个根
t_inside = ray_sphere(center[None, :], np.array([[1.0, 0, 0]]), center, radius)
print(f'从球心出发的光线 t = {t_inside[0]:.4f} (== 半径)')
t_pl = ray_plane(np.array([[0.0, 2.0, 0.0]]), normalize(np.array([[0.3, -1.0, -0.2]])), np.zeros(3), np.array([0.0, 1.0, 0.0]))
print(f'光线和地面 y = 0 的交点 y 坐标 = {(np.array([0.0, 2.0, 0.0]) + t_pl[0] * normalize(np.array([0.3, -1.0, -0.2])))[1]:.2e}')

<a id="ray-tracer"></a>

$$
\text{一个小型光线追踪器 (Ray Tracer)} \\
\text{1. 主光线 (Primary Ray): 从相机穿过每个像素中心发出一条光线 (方向由视场角算出, 和透视投影是反过来的过程)} \\
\quad d = \text{normalize}(x_{ndc} \tan\tfrac{fov}{2} \cdot aspect, \ y_{ndc} \tan\tfrac{fov}{2}, \ -1) \\
\text{2. 和所有物体求交, 取 t 最小的那个 (这就是光线追踪里的 "深度测试")} \\
\text{3. 阴影光线 (Shadow Ray): 从交点向光源发一条光线, 如果在到达光源之前碰到了物体, 这个点在阴影里, 只保留环境光} \\
\quad \text{起点要沿法向量偏移一个很小的 } \varepsilon \text{, 否则浮点误差会让光线和自己所在的表面相交 (阴影痤疮 shadow acne)} \\
\text{4. 着色 (Shading): Blinn-Phong} \\
\text{5. 反射光线 (Reflection Ray): 对镜面材质沿 } r = d - 2 (d \cdot n) n \text{ 再追踪一次, 递归, 结果按反射率混合}
$$

In [ ]:
W, H = 160, 120
fov = 60.0
aspect = W / H
eye = np.array([0.0, 1.2, 3.0])
light_pos = np.array([5.0, 6.0, 3.0])

# 物体: (类型, 参数, 颜色, 反射率)
spheres = [
    (np.array([0.0, 1.0, -3.0]), 1.0, np.array([0.9, 0.2, 0.2]), 0.0),
    (np.array([-2.3, 0.7, -4.0]), 0.7, np.array([0.8, 0.8, 0.8]), 0.6),    # 镜面球
    (np.array([2.0, 0.6, -2.3]), 0.6, np.array([0.2, 0.4, 0.9]), 0.0),
]
plane_p0, plane_n = np.zeros(3), np.array([0.0, 1.0, 0.0])

def plane_color(P_hit):
    check = (np.floor(P_hit[:, 0]) + np.floor(P_hit[:, 2])) % 2
    return np.where(check[:, None] == 0, [0.85, 0.85, 0.85], [0.35, 0.35, 0.35])

def nearest_hit(O, D):
    ts = [ray_sphere(O, D, c, r) for c, r, _, _ in spheres] + [ray_plane(O, D, plane_p0, plane_n)]
    ts = np.stack(ts)                       # (物体数, N)
    idx = np.argmin(ts, axis=0)
    return ts[idx, np.arange(len(O))], idx

def trace(O, D, depth=2):
    N = len(O)
    color = np.tile([0.6, 0.75, 0.95], (N, 1))        # 天空颜色
    t, idx = nearest_hit(O, D)
    hit = np.isfinite(t)
    if not hit.any():
        return color
    O_h, D_h, t_h, idx_h = O[hit], D[hit], t[hit], idx[hit]
    P_hit = O_h + t_h[:, None] * D_h

    n_obj = len(spheres)
    normal = np.tile(plane_n, (len(P_hit), 1))
    base = plane_color(P_hit)
    refl = np.zeros(len(P_hit))
    for k, (c, r, col, rf) in enumerate(spheres):
        sel = idx_h == k
        normal[sel] = (P_hit[sel] - c) / r
        base[sel] = col
        refl[sel] = rf

    L = normalize(light_pos - P_hit)
    V = -D_h
    Hh = normalize(L + V)
    # 阴影光线: 起点沿法向量偏移 epsilon
    shadow_O = P_hit + 1e-4 * normal
    t_shadow, _ = nearest_hit(shadow_O, L)
    dist_light = np.linalg.norm(light_pos - P_hit, axis=1)
    lit = t_shadow > dist_light

    ambient = 0.15 * base
    diffuse = 0.8 * base * np.maximum(np.sum(normal * L, axis=1), 0)[:, None]
    specular = 0.5 * np.maximum(np.sum(normal * Hh, axis=1), 0)[:, None]**80
    local = ambient + lit[:, None] * (diffuse + specular)

    if depth > 0 and np.any(refl > 0):
        m = refl > 0
        R_dir = D_h[m] - 2 * np.sum(D_h[m] * normal[m], axis=1, keepdims=True) * normal[m]
        reflected = trace(P_hit[m] + 1e-4 * normal[m], normalize(R_dir), depth - 1)
        local[m] = (1 - refl[m, None]) * local[m] + refl[m, None] * reflected
    color[hit] = np.clip(local, 0, 1)
    return color

# 主光线
px, py = np.meshgrid(np.arange(W) + 0.5, np.arange(H) + 0.5)
x_ndc = 2 * px / W - 1
y_ndc = 1 - 2 * py / H
tan_half = np.tan(np.deg2rad(fov) / 2)
dirs = normalize(np.stack([x_ndc * tan_half * aspect, y_ndc * tan_half, -np.ones_like(x_ndc)], axis=-1).reshape(-1, 3))
origins = np.tile(eye, (len(dirs), 1))

image = trace(origins, dirs).reshape(H, W, 3)
image_no_shadow_check, idx_all = nearest_hit(origins, dirs)
print(f'图像大小 = {image.shape}, 命中物体的像素比例 = {np.mean(np.isfinite(image_no_shadow_check)):.4f}')
for k in range(len(spheres)):
    print(f'球 {k} 占的像素数 = {np.sum((idx_all == k) & np.isfinite(image_no_shadow_check))}')

# 检查阴影: 红球正下方偏离光源一侧的地面点应该在阴影里
ground_pt = np.array([[-0.4, 0.0, -3.4]])
L_g = normalize(light_pos - ground_pt)
t_s, k_s = nearest_hit(ground_pt + 1e-4 * plane_n, L_g)
print(f'地面点 {ground_pt[0]} 到光源的阴影光线被物体 {k_s[0]} 挡住 ? {t_s[0] < np.linalg.norm(light_pos - ground_pt)}')

plt.figure(figsize=(4, 3))
plt.imshow(image)
plt.axis('off')
plt.title('numpy ray tracer')
plt.tight_layout()
plt.show()